<a href="https://colab.research.google.com/github/Keplerzhao/keplerzhao/blob/master/LPPLS_Model_Confidence_multi_processing_31_Oct_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
# 多核加速关键设置（防止子进程抢线程）
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import requests
from scipy.optimize import minimize
from multiprocessing import Pool, cpu_count
from tqdm.notebook import tqdm
from typing import List, Dict, Optional
import warnings
warnings.filterwarnings("ignore")

print(f"检测到 CPU 核心数: {cpu_count()}")

# ============================================================
# 1. 数据获取
# ============================================================
def get_close_prices(symbol: str, start_date: str,
                     api_key: str = '490ed0ef6adef36a548539c3e6da425e') -> pd.DataFrame:
    today = datetime.today().date()
    url = f"https://financialmodelingprep.com/api/v3/historical-price-full/{symbol}"
    params = {'from': start_date, 'to': today, 'apikey': api_key}
    response = requests.get(url, params=params, timeout=30)
    if response.status_code != 200:
        raise Exception(f"Request failed: {response.status_code}")
    data = response.json().get("historical", [])
    if not data:
        raise ValueError("No data found.")
    df = pd.DataFrame([{'date': item['date'], 'close': item['close']} for item in data])
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    df['log_price'] = np.log(df['close'].astype(float))
    return df


# ============================================================
# 2. 核心函数（降噪版）
# ============================================================
def lppls_func(t, tc, m, w, a, b, c1, c2):
    dt = np.abs(tc - t) + 1e-8
    return a + np.power(dt, m) * (b + c1*np.cos(w*np.log(dt)) + c2*np.sin(w*np.log(dt)))


def matrix_equation(t, y, tc, m, w):
    dt = np.abs(tc - t) + 1e-8
    phase = np.log(dt)
    fi = np.power(dt, m)
    gi = fi * np.cos(w * phase)
    hi = fi * np.sin(w * phase)
    X = np.column_stack([np.ones(len(t)), fi, gi, hi])
    try:
        beta = np.linalg.lstsq(X, y, rcond=None)[0]
    except:
        beta = np.zeros(4)
    return beta


def fit_single_window(t: np.ndarray, y: np.ndarray, max_searches: int = 18) -> Optional[Dict]:
    t1, t2 = float(t[0]), float(t[-1])
    dt = t2 - t1
    if dt < 80:
        return None

    bounds = [
        (t2 - 0.20*dt, t2 + 0.10*dt),
        (0.15, 0.85),
        (6.0, 13.0)
    ]

    best = None
    best_loss = np.inf

    for _ in range(max_searches):
        seed = np.array([np.random.uniform(lo, hi) for lo, hi in bounds])

        def obj(x):
            tc, m, w = x
            try:
                a, b, c1, c2 = matrix_equation(t, y, tc, m, w)
                pred = lppls_func(t, tc, m, w, a, b, c1, c2)
                return np.mean((pred - y)**2)
            except:
                return 1e12

        try:
            res = minimize(obj, seed, method='L-BFGS-B', bounds=bounds,
                           options={'maxiter': 60, 'ftol': 1e-9})
            if not res.success:
                continue

            tc, m, w = res.x
            a, b, c1, c2 = matrix_equation(t, y, tc, m, w)
            c = np.sqrt(c1**2 + c2**2 + 1e-12)

            O = (w / (2 * np.pi)) * np.log((tc - t1) / max(tc - t2, 1e-6))
            D = (m * abs(b)) / (w * c + 1e-12)

            pred = lppls_func(t, tc, m, w, a, b, c1, c2)
            rel_err = np.mean(np.abs(y - pred) / (np.abs(pred) + 1e-8))

            # 严格过滤（降噪）
            valid = (
                0.15 <= m <= 0.85 and
                6.0  <= w <= 13.0 and
                (t2 - 0.05*dt) <= tc <= (t2 + 0.08*dt) and
                O >= 2.5 and
                D >= 0.85 and
                rel_err <= 0.045 and
                abs(b) > 0.015
            )

            if valid and res.fun < best_loss:
                best_loss = res.fun
                best = {
                    "tc": tc, "m": m, "w": w,
                    "a": a, "b": b, "c1": c1, "c2": c2, "c": c,
                    "O": O, "D": D, "rel_err": rel_err,
                    "is_positive": b < 0,
                    "is_negative": b > 0
                }
        except:
            continue
    return best


# ============================================================
# 3. 多核 Worker
# ============================================================
def _worker(args):
    t2_idx, t_full, y_full, window_sizes, max_searches = args
    t2 = t_full[t2_idx]
    pos_cnt = neg_cnt = total = 0

    for dt in window_sizes:
        t1 = t2 - dt
        if t1 < t_full[0]:
            continue
        mask = (t_full >= t1) & (t_full <= t2)
        t_win = t_full[mask]
        y_win = y_full[mask]
        if len(t_win) < 80:
            continue

        total += 1
        fit = fit_single_window(t_win, y_win, max_searches)
        if fit is not None:
            if fit["is_positive"]:
                pos_cnt += 1
            elif fit["is_negative"]:
                neg_cnt += 1

    return {
        "t2_idx": t2_idx,
        "pos_conf": pos_cnt / total if total > 0 else 0.0,
        "neg_conf": neg_cnt / total if total > 0 else 0.0,
        "pos_valid": pos_cnt,
        "neg_valid": neg_cnt,
        "total": total
    }


def compute_confidence(dates, log_price, window_sizes=None, step=5,
                       max_searches=18, workers=None):
    if window_sizes is None:
        window_sizes = list(range(180, 521, 20))

    # 自动使用尽可能多的核心
    if workers is None:
        workers = max(1, cpu_count() - 1)   # 留一个核心给系统

    t = np.arange(len(log_price), dtype=np.float64)
    y = log_price.astype(np.float64)
    min_t2 = max(window_sizes) + 20
    t2_list = list(range(min_t2, len(t), step))

    print(f"多核加速计算中...")
    print(f"时间点: {len(t2_list)} | 窗口数: {len(window_sizes)} | searches: {max_searches} | workers: {workers}")

    args = [(idx, t, y, window_sizes, max_searches) for idx in t2_list]

    with Pool(processes=workers) as pool:
        results = list(tqdm(pool.imap(_worker, args), total=len(args), desc="Multi-core Computing"))

    records = []
    for r in results:
        idx = r["t2_idx"]
        records.append({
            "date": dates.iloc[idx],
            "t2": idx,
            "pos_confidence": r["pos_conf"],
            "neg_confidence": r["neg_conf"],
            "pos_valid": r["pos_valid"],
            "neg_valid": r["neg_valid"],
            "total_fits": r["total"]
        })
    return pd.DataFrame(records)


# ============================================================
# 4. 可视化
# ============================================================
def plot_confidence(price_df, conf_df, title="DS LPPL Confidence (Multi-core Denoised)"):
    fig, ax1 = plt.subplots(figsize=(14, 7))

    ax1.plot(price_df['date'], price_df['log_price'], color='tab:blue', lw=1.2, label='Log Price')
    ax1.set_ylabel('Log Price', color='tab:blue', fontsize=12)
    ax1.tick_params(axis='y', labelcolor='tab:blue')
    ax1.grid(True, alpha=0.3)

    ax2 = ax1.twinx()
    ax2.plot(conf_df['date'], conf_df['pos_confidence'], color='tab:red', lw=1.8, label='Positive Bubble')
    ax2.plot(conf_df['date'], conf_df['neg_confidence'], color='tab:green', lw=1.8, label='Negative Bubble')
    ax2.set_ylabel('DS LPPL Confidence', fontsize=12)

    max_conf = max(conf_df['pos_confidence'].max(), conf_df['neg_confidence'].max())
    ax2.set_ylim(0, max(0.55, max_conf * 1.15))

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

    plt.title(title, fontsize=14)
    fig.tight_layout()
    plt.show()


# ============================================================
# 5. 主程序（多核加速 + 降噪）
# ============================================================
if __name__ == "__main__":
    stock = "GLD"
    start = "2020-01-01"

    print(f"获取 {stock} 数据...")
    df = get_close_prices(stock, start)
    print(f"数据长度: {len(df)} | {df['date'].iloc[0].date()} → {df['date'].iloc[-1].date()}")

    # ---------- 多核推荐配置 ----------
    window_sizes = list(range(250, 500, 15))
    step = 5
    max_searches = 18
    workers = max(1, cpu_count() - 1)   # 自动使用多核
    # ----------------------------------

    conf_df = compute_confidence(
        dates=df['date'],
        log_price=df['log_price'].values,
        window_sizes=window_sizes,
        step=step,
        max_searches=max_searches,
        workers=workers
    )

    # 降噪平滑
    conf_df['pos_confidence'] = conf_df['pos_confidence'].rolling(window=6, min_periods=1, center=True).mean()
    conf_df['neg_confidence'] = conf_df['neg_confidence'].rolling(window=6, min_periods=1, center=True).mean()

    plot_confidence(df, conf_df, title=f"{stock} - DS LPPL Confidence (Multi-core + Denoised)")

    conf_df.to_csv(f"{stock.replace('^','')}_lppls_multicore.csv", index=False)
    print("多核加速结果已保存")

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import requests
from scipy.optimize import minimize
from multiprocessing import Pool, cpu_count
from tqdm.notebook import tqdm
from typing import List, Dict, Optional, Tuple
import warnings
warnings.filterwarnings("ignore")

print(f"检测到 CPU 核心数: {cpu_count()}")

# ============================================================
# 1. 数据获取
# ============================================================
def get_close_prices(symbol: str, start_date: str,
                     api_key: str = '490ed0ef6adef36a548539c3e6da425e') -> pd.DataFrame:
    today = datetime.today().date()
    url = f"https://financialmodelingprep.com/api/v3/historical-price-full/{symbol}"
    params = {'from': start_date, 'to': today, 'apikey': api_key}
    response = requests.get(url, params=params, timeout=30)
    if response.status_code != 200:
        raise Exception(f"Request failed: {response.status_code}")
    data = response.json().get("historical", [])
    if not data:
        raise ValueError("No data found.")
    df = pd.DataFrame([{'date': item['date'], 'close': item['close']} for item in data])
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    df['log_price'] = np.log(df['close'].astype(float))
    return df


# ============================================================
# 2. 核心 LPPL 函数
# ============================================================
def lppls_func(t, tc, m, w, a, b, c1, c2):
    dt = np.abs(tc - t) + 1e-8
    return a + np.power(dt, m) * (b + c1*np.cos(w*np.log(dt)) + c2*np.sin(w*np.log(dt)))


def matrix_equation(t, y, tc, m, w):
    dt = np.abs(tc - t) + 1e-8
    phase = np.log(dt)
    fi = np.power(dt, m)
    gi = fi * np.cos(w * phase)
    hi = fi * np.sin(w * phase)
    X = np.column_stack([np.ones(len(t)), fi, gi, hi])
    try:
        beta = np.linalg.lstsq(X, y, rcond=None)[0]
    except:
        beta = np.zeros(4)
    return beta


def compute_reliability(m, w, O, D, rel_err, b, tc, t2, dt) -> float:
    """
    论文启发的可靠性评分（0~1）
    综合参数合理性 + 拟合质量 + tc 位置
    分数越高，该拟合越值得信任
    """
    score = 1.0

    # 1. m 越靠近 0.3~0.7 越可靠
    if 0.3 <= m <= 0.7:
        score *= 1.0
    elif 0.2 <= m <= 0.8:
        score *= 0.7
    else:
        score *= 0.4

    # 2. ω 在经典区间更可靠
    if 6.5 <= w <= 11.5:
        score *= 1.0
    elif 6.0 <= w <= 13.0:
        score *= 0.75
    else:
        score *= 0.4

    # 3. O 和 D 越高越好
    score *= min(1.0, O / 4.0)
    score *= min(1.0, D / 1.2)

    # 4. 相对误差越小越好
    score *= max(0.1, 1.0 - rel_err * 12)

    # 5. |B| 不能太小
    score *= min(1.0, abs(b) / 0.05)

    # 6. tc 距离 t2 的位置（太近或太远都降分）
    tc_offset = (tc - t2) / dt
    if 0.0 <= tc_offset <= 0.06:
        score *= 1.0
    elif -0.03 <= tc_offset <= 0.10:
        score *= 0.7
    else:
        score *= 0.3

    return float(np.clip(score, 0.0, 1.0))


def fit_single_window(t: np.ndarray, y: np.ndarray, max_searches: int = 18) -> Optional[Dict]:
    t1, t2 = float(t[0]), float(t[-1])
    dt = t2 - t1
    if dt < 80:
        return None

    bounds = [
        (t2 - 0.20*dt, t2 + 0.10*dt),
        (0.15, 0.85),
        (6.0, 13.0)
    ]

    best = None
    best_loss = np.inf

    for _ in range(max_searches):
        seed = np.array([np.random.uniform(lo, hi) for lo, hi in bounds])

        def obj(x):
            tc, m, w = x
            try:
                a, b, c1, c2 = matrix_equation(t, y, tc, m, w)
                pred = lppls_func(t, tc, m, w, a, b, c1, c2)
                return np.mean((pred - y)**2)
            except:
                return 1e12

        try:
            res = minimize(obj, seed, method='L-BFGS-B', bounds=bounds,
                           options={'maxiter': 60, 'ftol': 1e-9})
            if not res.success:
                continue

            tc, m, w = res.x
            a, b, c1, c2 = matrix_equation(t, y, tc, m, w)
            c = np.sqrt(c1**2 + c2**2 + 1e-12)

            O = (w / (2 * np.pi)) * np.log((tc - t1) / max(tc - t2, 1e-6))
            D = (m * abs(b)) / (w * c + 1e-12)
            pred = lppls_func(t, tc, m, w, a, b, c1, c2)
            rel_err = np.mean(np.abs(y - pred) / (np.abs(pred) + 1e-8))

            # 基础过滤
            valid = (
                0.15 <= m <= 0.85 and
                6.0  <= w <= 13.0 and
                (t2 - 0.05*dt) <= tc <= (t2 + 0.08*dt) and
                O >= 2.5 and
                D >= 0.85 and
                rel_err <= 0.045 and
                abs(b) > 0.015
            )

            if valid and res.fun < best_loss:
                best_loss = res.fun
                reliability = compute_reliability(m, w, O, D, rel_err, b, tc, t2, dt)
                best = {
                    "tc": tc, "m": m, "w": w,
                    "a": a, "b": b, "c1": c1, "c2": c2, "c": c,
                    "O": O, "D": D, "rel_err": rel_err,
                    "reliability": reliability,
                    "is_positive": b < 0,
                    "is_negative": b > 0
                }
        except:
            continue
    return best


# ============================================================
# 3. 多核 Worker（返回更多信息）
# ============================================================
def _worker(args):
    t2_idx, t_full, y_full, window_sizes, max_searches = args
    t2 = t_full[t2_idx]

    pos_fits = []
    neg_fits = []
    total = 0

    for dt in window_sizes:
        t1 = t2 - dt
        if t1 < t_full[0]:
            continue
        mask = (t_full >= t1) & (t_full <= t2)
        t_win = t_full[mask]
        y_win = y_full[mask]
        if len(t_win) < 80:
            continue

        total += 1
        fit = fit_single_window(t_win, y_win, max_searches)
        if fit is not None:
            if fit["is_positive"]:
                pos_fits.append(fit)
            elif fit["is_negative"]:
                neg_fits.append(fit)

    # 原始 Confidence
    pos_conf = len(pos_fits) / total if total > 0 else 0.0
    neg_conf = len(neg_fits) / total if total > 0 else 0.0

    # 可靠性加权 Confidence（论文核心思想）
    pos_rel = np.mean([f["reliability"] for f in pos_fits]) if pos_fits else 0.0
    neg_rel = np.mean([f["reliability"] for f in neg_fits]) if neg_fits else 0.0

    pos_weighted = pos_conf * pos_rel
    neg_weighted = neg_conf * neg_rel

    # 平均 tc（用于 DTCAI）
    pos_tc = np.mean([f["tc"] for f in pos_fits]) if pos_fits else np.nan
    neg_tc = np.mean([f["tc"] for f in neg_fits]) if neg_fits else np.nan

    return {
        "t2_idx": t2_idx,
        "pos_conf": pos_conf,
        "neg_conf": neg_conf,
        "pos_rel": pos_rel,
        "neg_rel": neg_rel,
        "pos_weighted": pos_weighted,
        "neg_weighted": neg_weighted,
        "pos_tc": pos_tc,
        "neg_tc": neg_tc,
        "pos_valid": len(pos_fits),
        "neg_valid": len(neg_fits),
        "total": total
    }


def compute_indicators(dates, log_price, window_sizes=None, step=5,
                       max_searches=18, workers=None) -> pd.DataFrame:
    if window_sizes is None:
        window_sizes = list(range(250, 501, 15))
    if workers is None:
        workers = max(1, cpu_count() - 1)

    t = np.arange(len(log_price), dtype=np.float64)
    y = log_price.astype(np.float64)
    min_t2 = max(window_sizes) + 20
    t2_list = list(range(min_t2, len(t), step))

    print(f"多核计算中（含可靠性评分）...")
    print(f"时间点: {len(t2_list)} | 窗口数: {len(window_sizes)} | searches: {max_searches} | workers: {workers}")

    args = [(idx, t, y, window_sizes, max_searches) for idx in t2_list]

    with Pool(processes=workers) as pool:
        results = list(tqdm(pool.imap(_worker, args), total=len(args), desc="Computing"))

    records = []
    for r in results:
        idx = r["t2_idx"]
        # 简化 DTCAI：距离临界时间的天数（越小风险越高），再乘可靠性
        dtc_pos = (r["pos_tc"] - idx) if not np.isnan(r["pos_tc"]) else np.nan
        dtc_neg = (r["neg_tc"] - idx) if not np.isnan(r["neg_tc"]) else np.nan

        records.append({
            "date": dates.iloc[idx],
            "t2": idx,
            "pos_confidence": r["pos_conf"],
            "neg_confidence": r["neg_conf"],
            "pos_reliability": r["pos_rel"],
            "neg_reliability": r["neg_rel"],
            "pos_weighted": r["pos_weighted"],
            "neg_weighted": r["neg_weighted"],
            "pos_dtc": dtc_pos,
            "neg_dtc": dtc_neg,
            "pos_valid": r["pos_valid"],
            "neg_valid": r["neg_valid"],
            "total_fits": r["total"]
        })
    return pd.DataFrame(records)


# ============================================================
# 4. 可视化
# ============================================================
def plot_indicators(price_df, ind_df, title="LPPL Indicators (AI-inspired Reliability)"):
    fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

    # ------ 上图：价格 + 加权 Confidence ------
    ax1 = axes[0]
    ax1.plot(price_df['date'], price_df['log_price'], color='tab:blue', lw=1.2, label='Log Price')
    ax1.set_ylabel('Log Price', color='tab:blue')
    ax1.tick_params(axis='y', labelcolor='tab:blue')
    ax1.grid(True, alpha=0.3)

    ax1b = ax1.twinx()
    ax1b.plot(ind_df['date'], ind_df['pos_weighted'], color='tab:red', lw=1.8, label='Pos Weighted Conf')
    ax1b.plot(ind_df['date'], ind_df['neg_weighted'], color='tab:green', lw=1.8, label='Neg Weighted Conf')
    ax1b.set_ylabel('Weighted Confidence (Conf × Reliability)')
    ax1b.set_ylim(0, max(0.4, ind_df[['pos_weighted','neg_weighted']].max().max()*1.2))

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax1b.get_legend_handles_labels()
    ax1.legend(lines1+lines2, labels1+labels2, loc='upper left')
    ax1.set_title(title)

    # ------ 下图：原始 Confidence vs 可靠性 ------
    ax2 = axes[1]
    ax2.plot(ind_df['date'], ind_df['pos_confidence'], color='tab:red', lw=1.2, alpha=0.6, label='Pos Conf (raw)')
    ax2.plot(ind_df['date'], ind_df['neg_confidence'], color='tab:green', lw=1.2, alpha=0.6, label='Neg Conf (raw)')
    ax2.plot(ind_df['date'], ind_df['pos_reliability'], color='tab:red', lw=1.5, linestyle='--', label='Pos Reliability')
    ax2.plot(ind_df['date'], ind_df['neg_reliability'], color='tab:green', lw=1.5, linestyle='--', label='Neg Reliability')
    ax2.set_ylabel('Confidence / Reliability')
    ax2.set_ylim(0, 1.05)
    ax2.legend(loc='upper left')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


# ============================================================
# 5. 主程序
# ============================================================
if __name__ == "__main__":
    stock = "GLD"
    start = "2022-01-01"

    print(f"获取 {stock} 数据...")
    df = get_close_prices(stock, start)
    print(f"数据长度: {len(df)} | {df['date'].iloc[0].date()} → {df['date'].iloc[-1].date()}")

    # ---------- 推荐配置（250-500，步长15） ----------
    window_sizes = list(range(250, 501, 15))
    step = 5
    max_searches = 15
    workers = max(1, cpu_count() - 1)
    # ------------------------------------------------

    ind_df = compute_indicators(
        dates=df['date'],
        log_price=df['log_price'].values,
        window_sizes=window_sizes,
        step=step,
        max_searches=max_searches,
        workers=workers
    )

    # 平滑
    for col in ['pos_confidence', 'neg_confidence', 'pos_weighted', 'neg_weighted',
                'pos_reliability', 'neg_reliability']:
        ind_df[col] = ind_df[col].rolling(window=5, min_periods=1, center=True).mean()

    plot_indicators(df, ind_df, title=f"{stock} - LPPL with AI-inspired Reliability")

    ind_df.to_csv(f"{stock}_lppls_reliability.csv", index=False)
    print("结果已保存（含加权Confidence + 可靠性）")